In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.classification import *
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\LigaMX\\datasets\\LigaMX_dataset_v2.csv")

In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Equipo"])

In [4]:
df.head()

,Date,Time,Round,Day,Venue,Opponent,Referee,Equipo,Torneo,Temporada,...,TeamAwayForm5,OpponentForm5,OpponentWinRate5,OpponentSeasonPts,OpponentHomeForm5,OpponentAwayForm5,H2HWinRate,H2HLast5,FormDiff,SeasonPtsDiff
0,2014-07-18,19:30,Jornada 1,Fri,Away,Tijuana,Luis Santander,Puebla,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2014-07-18,19:30,Jornada 1,Fri,Away,Queretaro,Erick Yair Miranda,Pumas UNAM,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,2014-07-18,19:30,Jornada 1,Fri,Home,Pumas UNAM,Erick Yair Miranda,Queretaro,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
3,2014-07-18,19:30,Jornada 1,Fri,Home,Puebla,Luis Santander,Tijuana,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
4,2014-07-19,21:00,Jornada 1,Sat,Home,Tigres UANL,Erim Ramírez,Atlas,Apertura,2014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
drop_cols = [
    "Date", "Time", "Round", "Day", "Venue", "Opponent", "Referee", "Equipo", "Torneo", "Temporada", "Result", "Points"
]

cutoff = df["Date"].quantile(0.95)
data = df[df["Date"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["Date"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["ResultID"].sort_values())

OponentElo          -0.111290
OpponentAwayForm5   -0.028225
DayID               -0.025453
OpponentSeasonPts   -0.025366
OpponentForm5       -0.018891
OpponentWinRate5    -0.017448
TemporadaID         -0.008711
TorneoID            -0.008469
RoundID             -0.005759
TeamID              -0.000310
OpponentHomeForm5    0.000894
RefereeID            0.007649
TimeID               0.014496
TeamSeasonPts        0.020128
TeamHomeForm5        0.026066
OpponentID           0.029431
TeamWinRate5         0.049675
TeamAwayForm5        0.053529
H2HWinRate           0.054620
TeamForm5            0.057766
FormDiff             0.059044
SeasonPtsDiff        0.072725
VenueID              0.108383
H2HLast5             0.123514
TeamElo              0.141042
EloDiff              0.181458
ResultID             1.000000
Name: ResultID, dtype: float64


In [6]:
df = df.drop(columns=drop_cols)

In [7]:
print(df.columns.tolist())

['VenueID', 'OpponentID', 'TeamID', 'RefereeID', 'RoundID', 'TemporadaID', 'TorneoID', 'TimeID', 'DayID', 'ResultID', 'TeamElo', 'OponentElo', 'EloDiff', 'TeamForm5', 'TeamWinRate5', 'TeamSeasonPts', 'TeamHomeForm5', 'TeamAwayForm5', 'OpponentForm5', 'OpponentWinRate5', 'OpponentSeasonPts', 'OpponentHomeForm5', 'OpponentAwayForm5', 'H2HWinRate', 'H2HLast5', 'FormDiff', 'SeasonPtsDiff']


In [8]:
exp = setup(
    data=data, 
    target="ResultID", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    data_split_stratify=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,ResultID
2,Target type,Multiclass
3,Original data shape,"(6964, 27)"
4,Transformed data shape,"(6964, 27)"
5,Transformed train set shape,"(4874, 27)"
6,Transformed test set shape,"(2090, 27)"
7,Numeric features,26
8,Rows with missing values,5.5%
9,Preprocess,True


In [ ]:
compare_models()

In [9]:
cat = create_model('catboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4537,0.6149,0.4537,0.4636,0.4534,0.1597,0.1617
1,0.4853,0.6574,0.4853,0.4766,0.4537,0.2165,0.2282
2,0.4808,0.6535,0.4808,0.4805,0.4806,0.2176,0.2176
3,0.4989,0.6459,0.4989,0.4931,0.4919,0.2355,0.2373
4,0.4944,0.6736,0.4944,0.4755,0.4804,0.2295,0.2318
5,0.4605,0.6523,0.4605,0.4477,0.4523,0.1780,0.1788
6,0.5056,0.6839,0.5056,0.5050,0.5052,0.2391,0.2391
7,0.5011,0.7031,0.5011,0.4788,0.4837,0.2225,0.2255
8,0.5643,0.7343,0.5643,0.5564,0.5457,0.3165,0.3231


In [10]:
cat_tune = tune_model(cat)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4560,0.6242,0.4560,0.4641,0.4508,0.1628,0.1662
1,0.4718,0.6694,0.4718,0.4706,0.4143,0.1922,0.2148
2,0.4966,0.6733,0.4966,0.4854,0.4881,0.2360,0.2375
3,0.4921,0.6730,0.4921,0.4868,0.4883,0.2289,0.2295
4,0.4898,0.6911,0.4898,0.4689,0.4739,0.2220,0.2248
5,0.4989,0.6647,0.4989,0.4656,0.4702,0.2275,0.2337
6,0.5508,0.7241,0.5508,0.5239,0.5311,0.2918,0.2956
7,0.5192,0.7128,0.5192,0.4966,0.4887,0.2423,0.2509
8,0.5801,0.7276,0.5801,0.5763,0.5490,0.3359,0.3482


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(cat_tune);
#predict_model(cat);

In [ ]:
newpred = predict_model(cat_tune, data=data_unseen)
#newpred = predict_model(cat, data=data_unseen)

In [ ]:
rf = create_model('rf')

In [ ]:
rf_tune = tune_model(rf)

In [ ]:
predict_model(rf_tune);
#predict_model(rf);

In [ ]:
newpred = predict_model(rf_tune, data=data_unseen)
#newpred = predict_model(rf, data=data_unseen)

In [11]:
lgbm = create_model('lightgbm')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4221,0.5980,0.4221,0.4425,0.4268,0.1136,0.1151
1,0.4447,0.6221,0.4447,0.4508,0.4213,0.1558,0.1666
2,0.4763,0.6576,0.4763,0.4726,0.4742,0.2093,0.2094
3,0.4605,0.6401,0.4605,0.4495,0.4530,0.1796,0.1805
4,0.4740,0.6657,0.4740,0.4571,0.4633,0.2006,0.2017
5,0.4650,0.6478,0.4650,0.4439,0.4476,0.1790,0.1819
6,0.4740,0.6849,0.4740,0.4670,0.4703,0.1868,0.1870
7,0.5237,0.7118,0.5237,0.5060,0.5081,0.2580,0.2614
8,0.5892,0.7529,0.5892,0.5773,0.5673,0.3548,0.3628


In [ ]:
lgbm_tune = tune_model(lgbm)

In [ ]:
#predict_model(lgbm_tune);
predict_model(lgbm);

In [ ]:
#newpred = predict_model(lgbm_tune, data=data_unseen)
newpred = predict_model(lgbm, data=data_unseen)

In [12]:
et = create_model('et')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4582,0.6217,0.4582,0.4622,0.4562,0.1620,0.1635
1,0.4628,0.6434,0.4628,0.4803,0.4166,0.1793,0.2030
2,0.4786,0.6445,0.4786,0.4677,0.4720,0.2106,0.2112
3,0.4695,0.6369,0.4695,0.4568,0.4476,0.1828,0.1887
4,0.5011,0.6620,0.5011,0.4699,0.4755,0.2360,0.2414
5,0.4921,0.6574,0.4921,0.4503,0.4538,0.2139,0.2229
6,0.5463,0.7191,0.5463,0.5139,0.5243,0.2843,0.2881
7,0.5327,0.7042,0.5327,0.5094,0.4968,0.2612,0.2718
8,0.5553,0.7332,0.5553,0.5600,0.5247,0.2956,0.3076


In [ ]:
et_tune = tune_model(et)

In [ ]:
predict_model(et_tune);
#predict_model(et);

In [ ]:
#newpred = predict_model(et_tune, data=data_unseen)
newpred = predict_model(et, data=data_unseen)

In [ ]:
xgb = create_model('xgboost')

In [ ]:
xgb_tune = tune_model(xgb)

In [ ]:
#predict_model(xgb_tune);
predict_model(xgb);

In [ ]:
#newpred = predict_model(xgb_tune, data=data_unseen)
newpred = predict_model(xgb, data=data_unseen)

In [ ]:
nb = create_model('nb')

In [ ]:
nb_tune = tune_model(nb)

In [ ]:
#predict_model(nb_tune);
predict_model(nb);

In [ ]:
#newpred = predict_model(nb_tune, data=data_unseen)
newpred = predict_model(nb, data=data_unseen)

In [ ]:
blend1 = blend_models([cat_tune, lgbm])

In [ ]:
blend1_tune = tune_model(blend1)

In [ ]:
predict_model(blend1_tune);
#predict_model(blend1);

In [ ]:
#newpred = predict_model(blend1_tune, data=data_unseen)
newpred = predict_model(blend1, data=data_unseen)

In [ ]:
blend2 = blend_models([cat_tune, et])

In [ ]:
blend2_tune = tune_model(blend2)

In [ ]:
#predict_model(blend2_tune);
predict_model(blend2);

In [ ]:
#newpred = predict_model(blend2_tune, data=data_unseen)
newpred = predict_model(blend2, data=data_unseen)

In [ ]:
blend3 = blend_models([cat_tune, nb])

In [ ]:
blend3_tune = tune_model(blend3)

In [ ]:
predict_model(blend3_tune);
#predict_model(blend3);

In [ ]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

In [ ]:
blend4 = blend_models([cat_tune, et, nb])

In [ ]:
blend4_tune = tune_model(blend4)

In [ ]:
predict_model(blend4_tune);
#predict_model(blend4);

In [ ]:
newpred = predict_model(blend4_tune, data=data_unseen)
#newpred = predict_model(blend4, data=data_unseen)

In [13]:
blend5 = blend_models([cat_tune, lgbm, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4515,0.6192,0.4515,0.4607,0.4489,0.1534,0.1560
1,0.4673,0.6508,0.4673,0.4733,0.4257,0.1871,0.2076
2,0.4853,0.6674,0.4853,0.4761,0.4796,0.2209,0.2214
3,0.4966,0.6582,0.4966,0.4814,0.4849,0.2313,0.2333
4,0.4989,0.6831,0.4989,0.4771,0.4817,0.2350,0.2383
5,0.5011,0.6635,0.5011,0.4695,0.4715,0.2302,0.2372
6,0.5079,0.7157,0.5079,0.4787,0.4911,0.2290,0.2306
7,0.5418,0.7215,0.5418,0.5149,0.5117,0.2789,0.2871
8,0.5982,0.7546,0.5982,0.5941,0.5651,0.3641,0.3780


In [ ]:
blend5_tune = tune_model(blend5)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,20:56:21
Status,. . . . . . . . . . . . . . . . . .,Searching Hyperparameters
Estimator,. . . . . . . . . . . . . . . . . .,Voting Classifier


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
#predict_model(blend5_tune);
predict_model(blend5);

In [ ]:
newpred = predict_model(blend5_tune, data=data_unseen)
#newpred = predict_model(blend5, data=data_unseen)

In [ ]:
blend6 = blend_models([rf, et])

In [ ]:
blend6_tune = tune_model(blend6)

In [ ]:
#predict_model(blend6_tune);
predict_model(blend6);

In [ ]:
#newpred = predict_model(blend6_tune, data=data_unseen)
newpred = predict_model(blend6, data=data_unseen)

In [ ]:
blend7 = blend_models([lgbm, xgb])

In [ ]:
blend7_tune = tune_model(blend7)

In [ ]:
predict_model(blend7_tune);
#predict_model(blend7);

In [ ]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

In [ ]:
blend8 = blend_models([cat_tune, rf, lgbm, et, xgb, nb])

In [ ]:
blend8_tune = tune_model(blend8)

In [ ]:
predict_model(blend8_tune);
#predict_model(blend8);

In [ ]:
#newpred = predict_model(blend8_tune, data=data_unseen)
newpred = predict_model(blend8, data=data_unseen)

In [ ]:
save_model(blend5_tune, "LigaMX_model_v2")